### Import Dependencies

In [2]:
import yaml
from jinja2 import Template
from langsmith import Client

### RAG pipeline prompt

In [3]:
def build_prompt(question, formatted_context):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {formatted_context}

    Question:
    {question}
    """
    return prompt


### Jinja templates

In [4]:
formatted_context = "US"
question = "What is the capital of US?"

In [9]:
jinja_template = """
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {{formatted_context}}

    Question:
    {{question}}
    """

In [10]:
template = Template(jinja_template)

In [12]:
rendered_template = template.render(formatted_context="Washington DC", question="What is the capital of US?")

In [14]:
print(rendered_template)


    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    Washington DC

    Question:
    What is the capital of US?
    


In [15]:
def build_prompt_with_jinja(formatted_context, question):
    jinja_template = """
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {{formatted_context}}

    Question:
    {{question}}
    """

    template = Template(jinja_template)
    rendered_template = template.render(formatted_context=formatted_context, question=question)
    return rendered_template

In [17]:
print(build_prompt_with_jinja("Washington DC", "What is the capital of US?"))


    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    Washington DC

    Question:
    What is the capital of US?
    


### Load template from yaml file

In [19]:
def prompt_template_config(yaml_path, prompt_key):
    with open(yaml_path, "r") as f:
        config = yaml.safe_load(f)

    template_content = config['prompts'][prompt_key]
    template = Template(template_content)
    
    return template

In [25]:
template = prompt_template_config("prompts/retrieval_generation.yml", "retrieval_generation")

In [28]:
rendered_prompt = template.render(formatted_context="Washington DC", question="What is the capital of US?")

In [29]:
print(rendered_prompt)

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting

Context:
Washington DC

Question:
What is the capital of US?


In [31]:
def build_prompt_with_jinja(formatted_context, question):
    template = prompt_template_config("prompts/retrieval_generation.yml", "retrieval_generation")
    rendered_prompt = template.render(formatted_context=formatted_context, question=question)
    return rendered_prompt

In [32]:
print(build_prompt_with_jinja("Washington DC", "What is the capital of US?"))

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting

Context:
Washington DC

Question:
What is the capital of US?


### Prompt registry with Langsmith

In [34]:
ls_client = Client()

In [37]:
ls_template = ls_client.pull_prompt("retrieval-generation")

In [43]:
print(ls_template.messages[0].prompt.template)

You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.

Instructions:
- Answer the question based on the context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting

Context:
{{formatted_context}}

Question:
{{question}}


In [44]:
def prompt_template_registry(prompt_name):
    ls_client = Client()
    template_content = ls_client.pull_prompt(prompt_name).messages[0].prompt.template
    template = Template(template_content)
    return template

In [47]:
print(prompt_template_registry("retrieval-generation").render(formatted_context="Washington DC", question="What is the capital of US?"))

You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.

Instructions:
- Answer the question based on the context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting

Context:
Washington DC

Question:
What is the capital of US?
